# dARK Core Minter API - Smoke and Authority Tests

Focused checks for service reachability, worker status, and authority authorization endpoints.


**Contract update (2026-02):**\n- `alternate_identifiers` and `alternate_urls` must be sent inside `minimal_metadata` in `PUT /api/v1/arks/{ark}`.\n- `POST /api/v1/arks` and `POST /api/v1/arks/batch` no longer accept top-level alternate identifiers.\n

In [ ]:
import json
import os
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pprint import pprint

import requests

# -----------------------------------------------------------------------------
# Runtime configuration
# -----------------------------------------------------------------------------
MINTER_BASE_URL = os.getenv("MINTER_BASE_URL", "http://localhost:8001").rstrip("/")
MINTER_API_V1 = f"{MINTER_BASE_URL}/api/v1"
ADMIN_API_URL = os.getenv("ADMIN_API_URL", "http://localhost:8000/api/v1/admin").rstrip("/")

AUTHORITY_ID = os.getenv("AUTHORITY_ID", f"test-authority-{int(time.time())}")
NAAN = os.getenv("NAAN", "12345")

REGISTER_AUTHORITY = os.getenv("REGISTER_AUTHORITY", "true").lower() in {"1", "true", "yes", "y"}
EXISTING_CHAIN_ARK = os.getenv("EXISTING_CHAIN_ARK", "").strip() or None

POLL_INTERVAL_SECONDS = float(os.getenv("POLL_INTERVAL_SECONDS", "2"))
POLL_TIMEOUT_SECONDS = int(os.getenv("POLL_TIMEOUT_SECONDS", "90"))

print("MINTER_BASE_URL:", MINTER_BASE_URL)
print("MINTER_API_V1:", MINTER_API_V1)
print("ADMIN_API_URL:", ADMIN_API_URL)
print("AUTHORITY_ID:", AUTHORITY_ID)
print("NAAN:", NAAN)
print("REGISTER_AUTHORITY:", REGISTER_AUTHORITY)
print("EXISTING_CHAIN_ARK:", EXISTING_CHAIN_ARK)
print("POLL_TIMEOUT_SECONDS:", POLL_TIMEOUT_SECONDS)

# -----------------------------------------------------------------------------
# Test result tracking
# -----------------------------------------------------------------------------
RESULTS = []
CONTEXT = {}


def record(name: str, status: str, detail: str = ""):
    status = status.upper()
    if status not in {"PASS", "FAIL", "SKIP"}:
        raise ValueError(f"Invalid status: {status}")
    RESULTS.append({"name": name, "status": status, "detail": detail})
    print(f"[{status}] {name}")
    if detail:
        print("       ", detail)


def check(name: str, condition: bool, ok_detail: str = "", fail_detail: str = ""):
    if condition:
        record(name, "PASS", ok_detail)
    else:
        record(name, "FAIL", fail_detail or "Condition evaluated to False")


def skip(name: str, detail: str = ""):
    record(name, "SKIP", detail)


def summarize_results():
    passed = sum(1 for r in RESULTS if r["status"] == "PASS")
    failed = sum(1 for r in RESULTS if r["status"] == "FAIL")
    skipped = sum(1 for r in RESULTS if r["status"] == "SKIP")
    total = len(RESULTS)

    print("\n=== TEST SUMMARY ===")
    print(f"Total checks : {total}")
    print(f"Passed       : {passed}")
    print(f"Failed       : {failed}")
    print(f"Skipped      : {skipped}")

    if failed:
        print("\nFailed checks:")
        for row in RESULTS:
            if row["status"] == "FAIL":
                print(f"- {row['name']}: {row['detail']}")

    return {"total": total, "passed": passed, "failed": failed, "skipped": skipped}


def _decode_body(resp: requests.Response):
    ctype = (resp.headers.get("content-type") or "").lower()
    if "application/json" in ctype:
        try:
            return resp.json()
        except Exception:
            return resp.text
    try:
        return resp.json()
    except Exception:
        return resp.text


def call_api(name: str, method: str, path: str, expected, base: str = MINTER_API_V1, timeout: int = 30, **kwargs):
    url = f"{base}{path}"
    exp = expected if isinstance(expected, (list, tuple, set)) else [expected]

    try:
        resp = requests.request(method=method.upper(), url=url, timeout=timeout, **kwargs)
    except Exception as exc:
        record(name, "FAIL", f"Request exception against {url}: {exc}")
        return None, None

    body = _decode_body(resp)
    ok = resp.status_code in exp
    detail = f"expected={list(exp)} got={resp.status_code} method={method.upper()} url={url}"
    record(name, "PASS" if ok else "FAIL", detail)

    print(f"Status: {resp.status_code}")
    if isinstance(body, dict):
        pprint(body)
    else:
        text = str(body)
        if len(text) > 1000:
            text = text[:1000] + "..."
        print(text)

    return resp, body


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def json_metadata(title: str, version: int = 1):
    return json.dumps(
        {
            "title": title,
            "version": version,
            "source": "notebook",
            "timestamp": now_iso(),
            "description": "Integration metadata payload",
        },
        ensure_ascii=True,
    )


def xml_metadata(title: str):
    return f'''<?xml version="1.0" encoding="UTF-8"?>
<oai_dc:dc xmlns:oai_dc="http://www.openarchives.org/OAI/2.0/oai_dc/"
           xmlns:dc="http://purl.org/dc/elements/1.1/">
  <dc:title>{title}</dc:title>
  <dc:creator>Notebook Integration Test</dc:creator>
  <dc:date>{now_iso()}</dc:date>
  <dc:description>XML metadata payload for integration test</dc:description>
</oai_dc:dc>'''

def minimal_metadata(
    title: str,
    version: int = 1,
    alternate_identifiers=None,
    alternate_urls=None,
):
    year = datetime.now(timezone.utc).year
    payload = {
        "title": title,
        "authors": ["Notebook Integration Test"],
        "year": year,
        "publisher": "Notebook Integration Test",
        "subjects": [f"version-{version}"],
    }
    if alternate_identifiers:
        payload["alternate_identifiers"] = alternate_identifiers
    if alternate_urls:
        payload["alternate_urls"] = alternate_urls
    return payload


def wait_for_state(ark: str, expected_state: str, timeout_seconds: int = POLL_TIMEOUT_SECONDS):
    started = time.time()
    while True:
        resp, body = call_api(
            name=f"Poll ARK state={expected_state}",
            method="GET",
            path=f"/arks/{ark}",
            expected=[200, 404],
        )
        if resp is not None and resp.status_code == 200 and isinstance(body, dict):
            if body.get("state") == expected_state:
                return True, body

        if time.time() - started >= timeout_seconds:
            return False, body

        time.sleep(POLL_INTERVAL_SECONDS)


## 1. Service Smoke Checks

This section validates connectivity to API endpoints before deeper lifecycle tests.


In [ ]:
# Health endpoint (root level)
health_resp, health_body = call_api(
    name="Health endpoint reachable",
    method="GET",
    path="/health",
    expected=[200, 503],
    base=MINTER_BASE_URL,
)

# Worker status endpoint (API v1)
worker_resp, worker_body = call_api(
    name="Worker status endpoint reachable",
    method="GET",
    path="/worker/status",
    expected=200,
)

worker_running = bool(isinstance(worker_body, dict) and worker_body.get("running"))
check(
    name="Worker status has expected shape",
    condition=isinstance(worker_body, dict) and "status" in worker_body,
    ok_detail=f"worker_running={worker_running}",
    fail_detail="/worker/status did not return expected JSON",
)

CONTEXT["worker_running"] = worker_running


## 2. Authority Setup and Authorization Endpoints

If `REGISTER_AUTHORITY=true`, this notebook attempts to register a fresh authority using the Admin API.
Then it validates authority-related Minter API endpoints.


In [ ]:
def _get_authz():
    try:
        r = requests.get(f"{MINTER_API_V1}/authority/{AUTHORITY_ID}/authorized/{NAAN}", timeout=30)
        return r.status_code, _decode_body(r)
    except Exception as exc:
        return None, str(exc)


def _wait_authorized(timeout_seconds: int = 90):
    started = time.time()
    last = None
    while time.time() - started <= timeout_seconds:
        status, body = _get_authz()
        last = (status, body)
        if status == 200 and isinstance(body, dict) and body.get("authorized") is True:
            return True, last
        time.sleep(2)
    return False, last


if REGISTER_AUTHORITY:
    payload = {
        "uuid": AUTHORITY_ID,
        "naans": [NAAN],
        "fund_amount_eth": 0.05,
    }

    register_ok = False
    register_detail = ""
    for attempt in range(1, 3):
        try:
            resp = requests.post(f"{ADMIN_API_URL}/authority", json=payload, timeout=180)
            body = _decode_body(resp)
            if resp.status_code in {200, 201, 409}:
                register_ok = True
                register_detail = f"status={resp.status_code} attempt={attempt}"
                break
            register_detail = f"status={resp.status_code} body={body} attempt={attempt}"
        except Exception as exc:
            register_detail = f"attempt={attempt} exception={exc}"
        time.sleep(2)

    record("Admin register authority", "PASS" if register_ok else "FAIL", register_detail)

    # If registration did not result in immediate authorization, enforce NAAN authorization.
    authorized_now, last_auth = _wait_authorized(timeout_seconds=30)
    if not authorized_now:
        try:
            auth_resp = requests.post(
                f"{ADMIN_API_URL}/authority/{AUTHORITY_ID}/authorize-naan",
                json={"naan": NAAN},
                timeout=120,
            )
            auth_body = _decode_body(auth_resp)
            record(
                "Admin authorize NAAN fallback",
                "PASS" if auth_resp.status_code == 200 else "FAIL",
                f"status={auth_resp.status_code} body={auth_body}",
            )
        except Exception as exc:
            record("Admin authorize NAAN fallback", "FAIL", str(exc))
else:
    skip("Admin register authority", "REGISTER_AUTHORITY=false")

# Validate authority endpoints on minter API
_, auth_body = call_api(
    name="Authority details",
    method="GET",
    path=f"/authority/{AUTHORITY_ID}",
    expected=200,
)

_, naans_body = call_api(
    name="Authority NAAN list",
    method="GET",
    path=f"/authority/{AUTHORITY_ID}/naans",
    expected=200,
)

authorized_ok, last_auth = _wait_authorized(timeout_seconds=90)
record(
    "Authority authorization check",
    "PASS" if authorized_ok else "FAIL",
    f"last_auth={last_auth}",
)

if authorized_ok:
    check(
        name="Authority is authorized for NAAN",
        condition=True,
        ok_detail=f"last_auth={last_auth}",
        fail_detail=f"last_auth={last_auth}",
    )
else:
    check(
        name="Authority is authorized for NAAN",
        condition=False,
        ok_detail="",
        fail_detail=f"Authorization did not become true. last_auth={last_auth}",
    )



## 14. Final Summary

This cell prints a consolidated pass/fail summary and raises an exception if any check failed.


In [ ]:
summary = summarize_results()

if summary["failed"] > 0:
    raise AssertionError(f"There are {summary['failed']} failed checks. Review notebook output.")
else:
    print("All checked scenarios passed.")
